In [2]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [5]:
stime = 144 # ms
nlinks = 2048 # 1024*2, uni-directional
nhosts = 3072
bw = 1342176000 # B per second
load_list = range(4,41,4)
seed_list = [1,2,3,4,5]
topologytype = 2
nswitches = 80
os = 1
k = 64
nintervals = 48 # maxinterval + 1
topologyfile = 'evaltopologyfiles/rrg_80_64.edgelist'
serverfile = 'evalserverfiles/rrg_3072_80_64.sv'
npfile = 'evalnetpathfiles/netpath_rrg_80_64_su3.np'

In [ ]:
# # generate connection_matrices file (1)
# unv1bytes = 0
# unv1file = f'{homedir}rawtrafficfiles/cluster_b'
# maxinterval = 0
# with open(unv1file, 'r') as f:
#     lines = f.readlines()
#     for line in lines:
#         tokens = line.split(',')
#         # 0,32,31,10500
#         # interval,fromserver,toserver,bytes
#         unv1bytes += int(tokens[3])
#         maxinterval = max(maxinterval, int(tokens[0]))
# print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')

unv1bytes 102904411500, maxinterval 47, fullload 395823808512.0, ratio 3.8465193351987637


In [ ]:
# # generate connection_matrices file (2)
# random.seed(0)
# for load in load_list:
#     totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
#     mult = totalbytes / unv1bytes
#     actualbytes = 0
#     cmfile = f'cmfiles/rrg_load{load}.cm'
#     with open(cmfile, 'w') as fw:
#         with open(unv1file, 'r') as fr:
#             lines = fr.readlines()
#             iline = 0
#             while actualbytes < totalbytes:
#                 line = lines[iline]
#                 tokens = line.split(',')
#                 interval = int(tokens[0])
#                 fromserver = int(tokens[1])
#                 toserver = int(tokens[2])
#                 multbytes = int(tokens[3])

#                 if fromserver >= nhosts or toserver >= nhosts:
#                     iline += 1
#                     if iline >= len(lines):
#                         iline = 0
#                         if mult-1>0:
#                             mult = mult-1
#                     continue

#                 if mult >= 1 or (random.random() < mult):
#                     multbytes = adjustbytesbymtu(multbytes)
    
#                     # generate random start time
#                     start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

#                     fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
#                     actualbytes += int(multbytes)

#                 iline += 1
#                 if iline >= len(lines):
#                     iline = 0
#                     if mult-1>0:
#                         mult = mult-1

#                     # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

#     print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')


load 4%, totalbytes 15832952340.48, unv1bytes 102904411500, mult 0.15386077340795054, actualbytes 15833011500
load 8%, totalbytes 31665904680.96, unv1bytes 102904411500, mult 0.3077215468159011, actualbytes 31665906000
load 12%, totalbytes 47498857021.44, unv1bytes 102904411500, mult 0.46158232022385165, actualbytes 47498865000
load 16%, totalbytes 63331809361.92, unv1bytes 102904411500, mult 0.6154430936318022, actualbytes 63331920000
load 20%, totalbytes 79164761702.4, unv1bytes 102904411500, mult 0.7693038670397526, actualbytes 79164772500
load 24%, totalbytes 94997714042.88, unv1bytes 102904411500, mult 0.9231646404477033, actualbytes 94997731500
load 28%, totalbytes 110830666383.36, unv1bytes 102904411500, mult 0.07702541385565387, actualbytes 110830671000
load 32%, totalbytes 126663618723.84, unv1bytes 102904411500, mult 0.23088618726360433, actualbytes 126663660000
load 36%, totalbytes 142496571064.32, unv1bytes 102904411500, mult 0.384746960671555, actualbytes 142497069000
load

In [6]:
# generate pathweight file (1)
interval_stime = stime / nintervals
with open('rrgsu3_generate_pwfiles.conf', 'w') as f:
    for load in load_list:
        cmfile = f'cmfiles/rrg_load{load}.cm'
        for interval in range(nintervals):
            flowstart = interval_stime * interval
            flowend = interval_stime * (interval + 1)
            varfile = f'{homedir}rawpathweightfiles/pathtraffic_rrg_{nhosts}_{nswitches}_{k}_su3_cluster_b_load{load}_interval{interval}.var'
            qvarfile = f'{homedir}rawpathweightfiles/pathweight_rrg_{nhosts}_{nswitches}_{k}_su3_cluster_b_load{load}_interval{interval}.var'
            f.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_main/cluster_b/)
python3 ../../../../pararun.py --conf rrgsu3_generate_pwfiles.conf --worker 100

In [7]:
# generate pathweight file (2)
intervaldict = dict() #{0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
intervaldict[0] = 0
for interval in range(1, nintervals):
    intervaldict[interval] = interval - 1
with open('rrgsu3_copy_pwfiles.conf', 'w') as f:
    for load in load_list:
        for interval in range(nintervals):
            fromfile = f'{homedir}rawpathweightfiles/pathweight_rrg_{nhosts}_{nswitches}_{k}_su3_cluster_b_load{load}_interval{intervaldict[interval]}.var'
            tofile = f'{homedir}experiments/nsdi26fall/eval_main/cluster_b/pwfiles/pathweight_rrg_su3_cluster_b_load{load}_interval{interval}.pw'
            f.write(f'cp {fromfile} {tofile}\n')

actually run the copy commands in datacentre/

In [8]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_main/cluster_b/run_rrgsu3.conf'
with open(conffile, 'w') as f:
    for seed in seed_list:
        for load in load_list:
            cmfile = f'experiments/nsdi26fall/eval_main/cluster_b/cmfiles/rrg_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_main/cluster_b/pwfiles/pathweight_rrg_su3_cluster_b_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_main/cluster_b/outfiles/rrgsu3_load{load}_seed{seed}.out'
            f.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_main/cluster_b/run_rrgsu3.conf --worker 25